## Import Libraries

In [1]:
import os
import librosa
import numpy as np
from pathlib import Path
import zipfile
import requests
from tqdm import tqdm
import soundfile as sf
import json
import time
import logging
import matplotlib.pyplot as plt
import seaborn as sns

# Setting up plotting
plt.style.use('default')
sns.set_palette('husl')

print(f"librosa version: {librosa.__version__}")

librosa version: 0.11.0


## Configuration and Directory Setup

In [2]:
class AudioConfig:
    Audio_Sample_Rate = 16000
    Audio_Duration = 6
    Audio_N_Mfcc = 20
    Audio_Hop_Length = 512
    Audio_N_FFT = 2048
    Audio_N_MELS = 128

config = AudioConfig()

# Set up directories
base_dir = Path('./multimodal_mental_health_data')  # Fixed typo in 'health'
raw_audio_dir = base_dir / 'raw_audio'
processed_audio_dir = base_dir / 'processed_audio'
metadata_dir = base_dir / 'metadata'

# Create directories
for directory in [raw_audio_dir, processed_audio_dir, metadata_dir]:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"Created/Verified: {directory}")

# RAVDESS_Emotions mappinf for experiments

ravdess_emotion = {
    1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
    5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'
}

print(f"Emotion Categories: {list(ravdess_emotion.values())}")
print(f"Current Config - sr: {config.Audio_Sample_Rate}Hz, duration: {config.Audio_Duration}s")

Created/Verified: multimodal_mental_health_data/raw_audio
Created/Verified: multimodal_mental_health_data/processed_audio
Created/Verified: multimodal_mental_health_data/metadata
Emotion Categories: ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
Current Config - sr: 16000Hz, duration: 6s


### Setup Logging

In [3]:
def get_project_logger(name):
    logger = logging.getLogger(name)
    if not logger.hasHandlers():
        handler = logging.StreamHandler()
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

logger = get_project_logger('audio_collector')

print(f"Emotion Categories: {list(ravdess_emotion.values())}")
print(f"Current Config - sr: {config.Audio_Sample_Rate}Hz, duration: {config.Audio_Duration}s")
print(f"Base directory: {base_dir}")

Emotion Categories: ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
Current Config - sr: 16000Hz, duration: 6s
Base directory: multimodal_mental_health_data


## Manual RAVDESS dataset setup and verification

In [4]:
print(f"Manual ravdess dataset setup")

def setup_manual_ravdess():
    ravdess_path = raw_audio_dir / 'RAVDESS'
    if not ravdess_path.exists():
        logger.error(f"RAVDESS directory not found at {ravdess_path}. Please download and extract manually.")
        return False
    
    audio_files = list(ravdess_path.glob('**/*.wav'))

    if not audio_files:
        logger.error(f"No audio files found in {ravdess_path}. Please check the extraction.")
        return False
    
    print(f"Found {len(audio_files)} audio files in RAVDESS dataset.")

    # Checking directory structure
    actors = list(ravdess_path.glob('Actor_*'))
    if actors:
        print(f"Found {len(actors)} actors in the dataset.")
        for i, actor_dir in enumerate(sorted(actors)[:5]):
            files_count = len(list(actor_dir.glob('*.wav')))
            print(f"{actor_dir.name}: {files_count} files")
        if len(actors) > 5:
            print(f" ... and {len(actors) - 5} more actors.")
    else:
        print("No actor directories found. Please check the dataset structure.")
    
    return audio_files

Manual ravdess dataset setup


In [5]:
audio_files = setup_manual_ravdess()

if audio_files:
    print(f"Success! Ready to process {len(audio_files)} ravdess files.")
    ravdess_ready = True
    datset_path = raw_audio_dir / 'RAVDESS'
else:
    print(f"Please set up the RAVDESS dataset manually as per instructions.")
    ravdess_ready = False


Found 1440 audio files in RAVDESS dataset.
Found 24 actors in the dataset.
Actor_01: 60 files
Actor_02: 60 files
Actor_03: 60 files
Actor_04: 60 files
Actor_05: 60 files
 ... and 19 more actors.
Success! Ready to process 1440 ravdess files.


## Filename Parsing

In [6]:
def parse_ravdess_filename(filename_str):
    # Ravdess format: Modality-Voice-Emotion-Intensity-Statement-Repetition-Actor.wav
    parts = filename_str.replace('.wav', '').split('-')
    if len(parts) != 7:
        return None
    
    return {
        'modality': parts[0],
        'vocal_channel': parts[1],
        'emotion_code': int(parts[2]),
        'emotion': ravdess_emotion.get(int(parts[2]), 'unknown'),
        'intensity': int(parts[3]),
        'statement': parts[4],
        'repetition': parts[5],
        'actor': int(parts[6])
    }

# Testing filenames
test_filenames = [
    "03-01-06-01-02-01-12.wav",  # fearful
    "03-01-03-02-01-01-01.wav",  # happy
    "03-01-05-01-01-02-07.wav",  # angry
    "03-01-01-01-01-01-15.wav"   # neutral
]

print("Testing ravdess filename:")
for filename in test_filenames:
    parsed = parse_ravdess_filename(filename)
    print(f"File: {filename}")
    print(f"  → Emotion: {parsed['emotion']} (code: {parsed['emotion_code']})")
    print(f"  → Actor: {parsed['actor']}, Intensity: {parsed['intensity']}")
    print()

Testing ravdess filename:
File: 03-01-06-01-02-01-12.wav
  → Emotion: fearful (code: 6)
  → Actor: 12, Intensity: 1

File: 03-01-03-02-01-01-01.wav
  → Emotion: happy (code: 3)
  → Actor: 1, Intensity: 2

File: 03-01-05-01-01-02-07.wav
  → Emotion: angry (code: 5)
  → Actor: 7, Intensity: 1

File: 03-01-01-01-01-01-15.wav
  → Emotion: neutral (code: 1)
  → Actor: 15, Intensity: 1



## RAVDESS Dataset Analysis

In [7]:
ravdess_path = raw_audio_dir / 'RAVDESS'
audio_files = list(ravdess_path.glob('**/*.wav'))

# Parse all filenames
parsed_data = []
parse_errors = 0

for file in tqdm(audio_files, desc="Parsing filenames"):
    parsed = parse_ravdess_filename(file.name)
    if parsed:
        parsed['filepath'] = file
        parsed_data.append(parsed)
    else:
        parse_errors += 1

print(f"Dataset statistics:")
print(f"Total Files: {len(audio_files)}")
print(f"Successfully Parsed: {len(parsed_data)}")
print(f"Parse Errors: {parse_errors}")

if not parsed_data:
    print("No valid parsed data available. Please check the dataset.")


        

Parsing filenames: 100%|██████████| 1440/1440 [00:00<00:00, 607197.93it/s]

Dataset statistics:
Total Files: 1440
Successfully Parsed: 1440
Parse Errors: 0


#### Emotion distribution

In [8]:
emotion_counts = {}
for data in parsed_data:
    emotion = data['emotion']
    emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1

print(f"Emotion Distribution:")
for emotion, count in sorted(emotion_counts.items()):
    print(f"{emotion}: {count} files")

Emotion Distribution:
angry: 192 files
calm: 192 files
disgust: 192 files
fearful: 192 files
happy: 192 files
neutral: 96 files
sad: 192 files
surprised: 192 files


#### Actors distribution

In [9]:
actor_count = {}
for data in parsed_data:
    actor = data['actor']
    actor_count[actor] = actor_count.get(actor, 0) + 1

print(f"Actor Distribution:")
for actor, count in sorted(actor_count.items()):
    print(f"Actor {actor}: {count} files")
    

Actor Distribution:
Actor 1: 60 files
Actor 2: 60 files
Actor 3: 60 files
Actor 4: 60 files
Actor 5: 60 files
Actor 6: 60 files
Actor 7: 60 files
Actor 8: 60 files
Actor 9: 60 files
Actor 10: 60 files
Actor 11: 60 files
Actor 12: 60 files
Actor 13: 60 files
Actor 14: 60 files
Actor 15: 60 files
Actor 16: 60 files
Actor 17: 60 files
Actor 18: 60 files
Actor 19: 60 files
Actor 20: 60 files
Actor 21: 60 files
Actor 22: 60 files
Actor 23: 60 files
Actor 24: 60 files


#### Intensity Counts

In [10]:
intensity_count = {}
for data in parsed_data:
    intensity = data['intensity']
    intensity_count[intensity] = intensity_count.get(intensity, 0) + 1

print(f"Intenstiy Distribution:")
for intensity, count in sorted(intensity_count.items()):
    intensity_name = "Normal" if intensity == 1 else "Strong"
    print(f"{intensity} ({intensity_name}): {count} files")


Intenstiy Distribution:
1 (Normal): 768 files
2 (Strong): 672 files


In [12]:
# Visualization
fig, axes = plt.subplot(1, 3, figsize=(18, 5))

# Emotion distribution
emotions = list(emotion_counts.keys())
counts = list(emotion_counts.values())
axes[0].bar(emotions, counts, color='skyblue')
axes[0].set_title('Emotion Distribution')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Number of Files')
axes[0].tick_params(axis='x', rotation=45)

# Actor distribution
actors = sorted(list(actor_count.keys()))
actor_file_counts = [actor_count[a] for a in actors]
axes[1].bar([f"A{a:02d}" for a in actors], actor_file_counts, color='lightgreen')
axes[1].set_title('Files per Actor')
axes[1].set_xlabel('Actor')
axes[1].set_ylabel('Number of Files')
axes[1].tick_params(axis='x', rotation=45)

# Intensity distribution
intensities = list(intensity_count.keys())
intensity_names = ["Normal" if i == 1 else "Strong" for i in intensities]
intensity_file_counts = list(intensity_count.values())
axes[2].bar(intensity_names, intensity_file_counts, color='lightgreen')
axes[2].set_title('Intensity Distribution')
axes[2].set_xlabel('Intensity')
axes[2].set_ylabel('Number of Files')
    
plt.tight_layout()
plt.show()

TypeError: subplot() takes 1 or 3 positional arguments but 2 were given

<Figure size 640x480 with 0 Axes>